 # Classification using embeddings (multi-label version)

This notebook allows to train multiple models with multiple settings on multiple datasets for the Abstract->Domain (Research Focus) classifiaction tasks.

Abstracts are embedded using [Qwen/Qwen3-Embedding-8B](https://huggingface.co/Qwen/Qwen3-Embedding-8B).

Several classification models can be tests:
* Multi-Layer Perceptron
* Random Forest
* Logistic Regression

Several stratisfaction strategies can be tested:
* None
* Cardinality: by number of labels per sample (for simple vs complex cases representation)
* Minority: by presence/absence of minor domains (for fair minor domains representation)


Expected dataset format (* indicate optional fields):
| hdp_id*  | appl_id  | project_num*          | abstract_text                                                        | domains                                |
|----------|----------|-----------------------|---------------------------------------------------------------------|----------------------------------------|
| HDP00033 | 10056337 | 1R61NS113258-01A1     | PROJECT SUMMARY. Taxanes are among the most efficacious…            | Depression, Pain Interference, Sleep, Substance Use |
| HDP00097 | 10157953 | 1R61NS118651-01A1     | PROJECT SUMMARY.  Chronic pain represents a public health…          | Anxiety, Substance Use                 |
| HDP00110 | 9870024  | 1UG3NR019196-01       | Chronic musculoskeletal pain (CMP) is the most common…              | Depression, Sleep, Substance Use       |



 How to read results:

| Scenario                              | Precision           | Recall             | F1                | Accuracy        |
|---------------------------------------|---------------------|--------------------|-------------------|-----------------|
| All labels were assign correct        | 1.0                 | 1.0                | 1.0               | 1.0             |
| Some labels correct, some missing     | 1.0                 | <1.0               | <1.0              | <1.0            |
| Some labels correct, some extra       | <1.0                | 1.0                | <1.0              | <1.0            |
| None correct labels                   | 0                   | 0                  | 0                 | 0               |

 ## Libraries

In [ ]:
# Uncomment these lines to install libraries used in this notebook
# Note: this combination of libraries works if pip version 22.0.2 or 26.0.1

#!pip install --no-cache-dir ipywidgets
#!pip install --no-cache-dir light-the-torch
#!ltt install --no-cache-dir torch torchvision
#!pip install --no-cache-dir transformers[torch]
#!pip install --no-cache-dir pandas scikit-learn matplotlib
#!pip install -q sentence-transformers
#!pip install -q chromadb
#!pip install tqdm

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, accuracy_score, recall_score, precision_score,
    classification_report, precision_recall_curve, average_precision_score,
)
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import chromadb
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import StandardScaler


 ## Inputs/outputs directories

In [ ]:
input_dir = "./inputs"
output_dir = "./outputs"
models_dir = os.path.join(output_dir, "models")
plots_dir = os.path.join(output_dir, "plots", "classification")
summary_plots_dir = os.path.join(output_dir, "plots", "summary")
reports_dir = os.path.join(output_dir, "reports")
embeddings_dir = os.path.join(output_dir, "embeddings")
chromadb_dir = os.path.join(embeddings_dir, "chromadb")

os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(summary_plots_dir, exist_ok=True)
os.makedirs(reports_dir, exist_ok=True)
os.makedirs(embeddings_dir, exist_ok=True)
os.makedirs(chromadb_dir, exist_ok=True)

 ## Inputs - training data

Specify path to your input datasets

In [ ]:
dataset_files = [
    ("heal-basic_full", os.path.join(input_dir, "heal_abstract_domain_basic_full.tsv")),
    ("heal-basic_collapsed", os.path.join(input_dir, "heal_abstract_domain_basic_collapsed.tsv")),
    #("heal-extended_full", os.path.join(input_dir, "heal_abstract_domain_extended_full.tsv")),
    #("heal-extended_collapsed", os.path.join(input_dir, "heal_abstract_domain_extended_collapsed.tsv")),
    #("nih-reporter_full", os.path.join(input_dir, "nih-reporter_abstract_domain_full.tsv")),
    #("nih-reporter_collapsed", os.path.join(input_dir, "nih-reporter_abstract_domain_collapsed.tsv")),
]

In [ ]:
# Check if any record is missing appl_id
for _, filename in dataset_files:
    df = pd.read_csv(filename, sep="\t")
    if "appl_id" in df.columns:
        missing_df = df[df["appl_id"].isnull() | (df["appl_id"] == "")]
        n_missing = missing_df.shape[0]
        print(f"{filename}: {n_missing} records missing appl_id")
        if n_missing > 0:
            print(missing_df.head())
    else:
        print(f"{filename}: NO appl_id column present")

 ## Helpers

In [ ]:
def save_embedding_to_chroma(abstract_id, embedding, collection):
    collection.add(
        ids=[str(abstract_id)],
        embeddings=[embedding.tolist()],
    )

In [ ]:
def get_embedding_from_chroma(abstract_id, collection):
    result = collection.get(ids=[str(abstract_id)], include=["embeddings"])
    embeddings = result.get("embeddings")
    if embeddings is not None and len(embeddings) > 0 and embeddings[0] is not None:
        return np.array(embeddings[0], dtype=np.float32)
    return None

In [ ]:
def get_embedding(text, tokenizer, model, device):
    inputs = tokenizer(
        text, truncation=True, padding=True, return_tensors="pt", max_length=512
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().float().numpy()
    return embeddings

In [ ]:
def encode_multilabel(df, domain2idx):
    n_labels = len(domain2idx)
    y = np.zeros((len(df), n_labels), dtype=int)
    for i, dom_str in enumerate(df["domains"].fillna("")):
        if dom_str == "":
            continue
        labels = [d.strip() for d in dom_str.split(",")]
        for d in labels:
            if d in domain2idx:
                y[i, domain2idx[d]] = 1
    return y


In [ ]:
def make_cardinality_stratify_labels(y, max_bins=5):
    cardinalities = y.sum(axis=1)
    binned = np.minimum(cardinalities, max_bins - 1).astype(int)
    return binned

In [ ]:
def make_minority_stratify_labels(y, domains, minority_domains):
    """
    Returns a stratification label:
    0 = none of the minority domains present
    1 = at least one minority domain present
    """
    minority_indices = [domains.index(d) for d in minority_domains if d in domains]
    if not minority_indices:
        return None
    minority_presence = (y[:, minority_indices].sum(axis=1) > 0).astype(int)
    return minority_presence

In [ ]:
def plot_model_scores(model_name, f1, acc, rec, prec, outdir="outputs/plots"):
    os.makedirs(outdir, exist_ok=True)
    plt.figure(figsize=(6, 4))
    metrics_names = ["F1", "Accuracy", "Recall", "Precision"]
    metrics_values = [f1, acc, rec, prec]
    plt.bar(metrics_names, metrics_values)
    plt.ylim(0, 1)
    plt.title(f"{model_name} Performance")
    plt.ylabel("Score")
    for i, v in enumerate(metrics_values):
        plt.text(i, v + 0.01, f"{v:.3f}", ha="center")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{model_name}_performance.png"))
    plt.close()

In [ ]:
def plot_multiclass_precision_recall(y_true, y_scores, domains, model_name, outdir="outputs/plots"):
    os.makedirs(outdir, exist_ok=True)
    n_classes = y_true.shape[1]
    plt.figure(figsize=(10, 8))
    for i in range(n_classes):
        if y_true[:, i].sum() == 0:
            continue
        precision, recall, _ = precision_recall_curve(y_true[:, i], y_scores[:, i])
        ap = average_precision_score(y_true[:, i], y_scores[:, i])
        plt.plot(recall, precision, lw=2, label=f"{domains[i]} (AP={ap:.2f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision–Recall curves for {model_name}")
    plt.legend(loc="best", fontsize="small")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{model_name}_precision_recall.png"))
    plt.close()


In [ ]:
def evaluate_and_plot(model, X_test, y_test, domains, model_name):
    if isinstance(model, MLPClassifier):
        proba = model.predict_proba(X_test)
        if isinstance(proba, list):
            y_scores = np.column_stack([p[:, 1] for p in proba])
        else:
            y_scores = proba
    elif isinstance(model, MultiOutputClassifier):
        proba_list = model.predict_proba(X_test)
        cols = []
        for p in proba_list:
            p = np.asarray(p)
            if p.shape[1] == 1:
                prob1 = np.zeros(p.shape[0])
            else:
                prob1 = p[:, 1]
            cols.append(prob1)
        y_scores = np.column_stack(cols)
    else:
        y_scores = model.predict(X_test)

    y_pred = (y_scores > 0.5).astype(int)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    report = classification_report(
        y_test, y_pred, target_names=domains, zero_division=0,
    )
    # Save report and plot
    with open(f"{reports_dir}/{model_name}_report.txt", "w") as f:
        f.write(report)
    plot_model_scores(model_name, f1, acc, rec, prec, outdir=plots_dir)
    plot_multiclass_precision_recall(y_test, y_scores, domains, model_name, outdir=plots_dir)
    print(f"{model_name}: F1={f1:.4f}, Acc={acc:.4f}, Rec={rec:.4f}, Prec={prec:.4f}")
    return {
        "model_name": model_name,
        "F1": f1,
        "Accuracy": acc,
        "Recall": rec,
        "Precision": prec,
    }

## Conditional Embedding

In [ ]:
# Change as needed
embedding_model = "Qwen/Qwen3-Embedding-8B"

In [ ]:
collection_name = f"abstract_embeddings_{embedding_model.replace('/', '_').replace('-', '_')}"
chroma_client = chromadb.PersistentClient(path=chromadb_dir)
collection = chroma_client.get_or_create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"},
)

If embedding database doesn't exit - it will be created, if it exists - existing database will be used to save GPU resources

In [ ]:
db_present = os.path.exists(chromadb_dir) and bool(os.listdir(chromadb_dir)) \
    and collection.count() > 0

if not db_present:
    print(f"[ChromaDB] Embeddings database (collection '{collection_name}') not found or is empty. Initializing embedding generation using GPU...")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(embedding_model, trust_remote_code=True)
    model = AutoModel.from_pretrained(embedding_model, trust_remote_code=True).to(device)
    model.eval()

    for ds_name, ds_file in dataset_files:
        print(f"[ChromaDB] Generating embeddings for dataset: {ds_name}")
        df = pd.read_csv(ds_file, sep="\t")
        df["abstract_text"] = df["abstract_text"].fillna("").astype(str)
        for idx, row in tqdm(df.iterrows(), total=df.shape[0], desc=f"{ds_name} embeddings"):
            abstract_id = row["appl_id"]
            emb = get_embedding_from_chroma(abstract_id, collection)
            if emb is None:
                emb = get_embedding(row["abstract_text"], tokenizer, model, device)
                save_embedding_to_chroma(abstract_id, emb, collection)
            
    print("[ChromaDB] All dataset embeddings generated and saved.")
else:
    print(f"[ChromaDB] Embeddings database found (collection '{collection_name}'). Will load embeddings from ChromaDB. No need for GPU/tokenizer.")


In [ ]:
# Check collection size and appl_id of interest
debug_appl_id = 10056337

print(f"[ChromaDB] Collection '{collection_name}' size:", collection.count())
print(f"Embedding for {debug_appl_id}:", get_embedding_from_chroma(str(debug_appl_id), collection))
    

 ## Load all domains

In [ ]:
all_domains = set()
for _, fname in dataset_files:
    df = pd.read_csv(fname, sep="\t")
    all_domains.update([d.strip() for ds in df["domains"].fillna("").str.split(",") for d in ds])
all_domains = sorted(all_domains)
domain2idx = {d: i for i, d in enumerate(all_domains)}

# Main training loop

In [ ]:
# Update stratification modes, seeds, test_size for training as needed
#strat_modes = ["no_strat", "cardinality_strat", "minority_strat"]
#seeds = [1, 21, 42, 123, 456, 789, 999, 2024, 30380]
strat_modes = ["no_strat"]
seeds = [123]
test_size = 0.2

In [ ]:
# Edit datasets as needed if you want to run training on specific dataset vs all datasets defined at the beginning of the notebook

#dataset_files = [
#    ("heal-basic_full", os.path.join(input_dir, "heal_abstract_domain_basic_full.tsv")),
#    ("heal-basic_collapsed", os.path.join(input_dir, "heal_abstract_domain_basic_collapsed.tsv")),
#    ("heal-extended_full", os.path.join(input_dir, "heal_abstract_domain_extended_full.tsv")),
#    ("heal-extended_collapsed", os.path.join(input_dir, "heal_abstract_domain_extended_collapsed.tsv")),
#    ("nih-reporter_full", os.path.join(input_dir, "nih-reporter_abstract_domain_full.tsv")),
#    ("nih-reporter_collapsed", os.path.join(input_dir, "nih-reporter_abstract_domain_collapsed.tsv")),
#]

In [ ]:
results = []

summary_path = os.path.join(output_dir, "classification_summary_all.tsv")

for dataset_name, dataset_file in dataset_files:
    print(f"\n=== Processing {dataset_name} ===")
    df = pd.read_csv(dataset_file, sep="\t")
    df["embedding"] = df["appl_id"].apply(lambda id: get_embedding_from_chroma(id, collection))
    missing = df["embedding"].isnull().sum()
    if missing > 0:
        print(f"[WARN] {missing} records missing embeddings in ChromaDB.")
    X = np.stack(df["embedding"].values)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    y = encode_multilabel(df, domain2idx).astype(int)

    for strat_mode in strat_modes:
        print(f"  -- Stratification mode: {strat_mode}")
        if strat_mode == "no_strat":
            strat_labels = None
        elif strat_mode == "cardinality_strat":
            strat_labels = make_cardinality_stratify_labels(y, max_bins=5)
        elif strat_mode == "minority_strat":
            minority_domains = ["Anxiety", "Depression", "Quality of Life"]
            strat_labels = make_minority_stratify_labels(y, all_domains, minority_domains)
        else:
            raise ValueError(strat_mode)

        for seed in seeds:
            print(f"    *** Seed: {seed} ***")
            if strat_labels is not None:
                try:
                    X_train, X_test, y_train, y_test = train_test_split(
                        X_scaled, y, test_size=test_size, random_state=seed, stratify=strat_labels
                    )
                except ValueError as e:
                    print(f"[SKIP] {dataset_name} {strat_mode} seed {seed}: {e}")
                    continue
            else:
                X_train, X_test, y_train, y_test = train_test_split(
                    X_scaled, y, test_size=test_size, random_state=seed
                )

            train_counts = y_train.sum(axis=0)
            valid_indices = np.where(train_counts > 0)[0]
            valid_domains = [all_domains[i] for i in valid_indices]

            models = [
                ("mlp", MLPClassifier(hidden_layer_sizes=(512,256), activation="relu", solver="adam",
                                        alpha=1e-4, learning_rate_init=1e-3, max_iter=200,
                                        early_stopping=True, validation_fraction=0.1, random_state=seed)),
                ("rf", MultiOutputClassifier(RandomForestClassifier(n_estimators=200, max_depth=None,
                                                n_jobs=-1, random_state=seed))),
                ("logreg", MultiOutputClassifier(LogisticRegression(max_iter=1000, solver="lbfgs",
                                                    random_state=seed))),
            ]
            
            for model_name, model_obj in models:
                print(f"      >> Training model: {model_name}")
                fit_failed = False
                try:
                    with warnings.catch_warnings():
                        warnings.filterwarnings("ignore", category=ConvergenceWarning)
                        if model_name in ("logreg", "lasso"):  # Handles logreg/lasso index masking
                            model_obj.fit(X_train, y_train[:, valid_indices])
                        else:
                            model_obj.fit(X_train, y_train)
                except Exception as e:
                    print(f"[SKIP] {dataset_name} {strat_mode} seed {seed} model {model_name}: {e}")
                    fit_failed = True

                if fit_failed:
                    metrics = {
                        "model_name": model_name,
                        "F1": np.nan,
                        "Accuracy": np.nan,
                        "Recall": np.nan,
                        "Precision": np.nan,
                        "dataset": dataset_name,
                        "strat_mode": strat_mode,
                        "threshold": "default",
                        "seed": seed,
                    }
                else:
                    report_name = f"{dataset_name}_{strat_mode}_{model_name}_seed{seed}"
                    if model_name in ("logreg", "lasso"):
                        metrics = evaluate_and_plot(
                            model_obj, X_test, y_test[:, valid_indices], valid_domains, report_name
                        )
                    else:
                        metrics = evaluate_and_plot(
                            model_obj, X_test, y_test, all_domains, report_name
                        )
                    model_path = os.path.join(models_dir, f"{report_name}.joblib")
                    joblib.dump(model_obj, model_path)
                    metrics["dataset"] = dataset_name
                    metrics["strat_mode"] = strat_mode
                    metrics["threshold"] = "default"
                    metrics["seed"] = seed
                results.append(metrics)
                # Write after every metrics append
                results_df = pd.DataFrame(results)
                results_df.to_csv(summary_path, sep="\t", index=False)

# Check results

In [ ]:
# Load processed summary file
summary_path = os.path.join(output_dir, "classification_summary_all.tsv")
df = pd.read_csv(summary_path, sep="\t")
df